# Module C: REPA Alignment (C4-C5)

**REPA-Style Intermediate Alignment for the JEPA Dynamics Predictor**

| # | Task | Detail |
|---|------|--------|
| C4 | REPA alignment | Cache DINOv2 intermediate layer features (all 12 blocks), add projection heads, train with $\lambda \in \{0, 0.1, 0.3, 0.5\}$ |
| C5 | REPA ablation results | Cosine sim, rollout drift, CEM success with/without REPA |

**Prerequisites**:
- Module A predictor checkpoint at `outputs/module_a_transformer_K1_T8.pt`
- Cached embeddings in `data/embeddings/` (`embeddings_{train,val}.pt`, `actions_{train,val}.pt`, `triplets_{train,val}_K1.pt`)
- LeRobot MP4 videos in `data/robomimic-ph-lift-image/videos/` (for DINOv2 intermediate feature caching)

> Split from `module_c_bc_repa.ipynb`. JEPA-augmented BC (C1-C3) is in `module_c_bc_jepa_aug.ipynb`.
>
> **Fixes applied vs original:**
> 1. REPA alignment target is the **predicted future frame** (`dino_inter[tpK_idx]`), not the input frame
> 2. Warm-started from Module A predictor checkpoint with **lambda=0 control** for fair ablation
> 3. Post-norm CLS features + fp32 REPA loss
> 4. Resumable feature caching (per-episode chunks)


## Section 1: Setup & Imports

In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch>=2.0', 'matplotlib', 'scikit-learn', 'tqdm', 'numpy', 'pandas', 'scipy', 'pyarrow', 'timm', 'av']:
    pip_install(pkg)

import os, math, random, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# -- Paths (Colab-aware) --
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/jepa_action")
    print("Mounted Google Drive. Make sure data/ is at MyDrive/jepa_action/data/")
except ImportError:
    ROOT = Path(os.getcwd()).resolve()
    if not (ROOT / "data" / "embeddings").exists():
        if (ROOT.parent / "data" / "embeddings").exists():
            ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
EMBED_DIR = DATA_DIR / "embeddings"
PARQUET_DIR = DATA_DIR / "robomimic-ph-lift-image" / "data" / "chunk-000"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config
class Config:
    # Core
    embed_dim = 384
    action_dim = 7
    d_model = 256
    n_heads = 4
    n_layers_tr = 4
    batch_size = 64
    lr = 1e-4
    weight_decay = 0.05
    warmup_steps = 200
    use_amp = True
    dropout_p = 0.2
    seed = 42
    K_values = [1, 5, 10, 20]
    best_T = 8
    n_train_episodes = 160
    rollout_K = 1  # for module_a checkpoint path
    # REPA
    dino_all_layers = 12
    repa_layer_map = [2, 5, 8, 11]
    repa_lambda_values = [0, 0.1, 0.3, 0.5]  # 0 = control (no REPA loss)
    repa_epochs = 50

cfg = Config()

# Device & reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

print('Setup complete.')


## Section 2: Load Cached Data

Load pre-computed embeddings, actions, and per-episode data from Parquet files. Build episode-to-embedding-index mapping.

In [ ]:
# Load cached embeddings and actions
train_emb = torch.load(EMBED_DIR / 'embeddings_train.pt').float()
val_emb = torch.load(EMBED_DIR / 'embeddings_val.pt').float()
train_act = torch.load(EMBED_DIR / 'actions_train.pt').float()
val_act = torch.load(EMBED_DIR / 'actions_val.pt').float()
norm_stats = torch.load(EMBED_DIR / 'norm_stats.pt')

print(f'Train embeddings: {train_emb.shape}')
print(f'Val embeddings:   {val_emb.shape}')
print(f'Train actions:    {train_act.shape}')

# Load all Parquet files for episode boundaries
parquet_files = sorted(PARQUET_DIR.glob('episode_*.parquet'))
print(f'Found {len(parquet_files)} parquet episodes')

all_ep_actions = []
all_ep_n_frames = []

for pf in tqdm(parquet_files, desc='Loading parquets'):
    df = pd.read_parquet(pf)
    actions_raw = np.stack(df['action'].values).astype(np.float32)
    all_ep_actions.append(torch.tensor(actions_raw[::2]))
    all_ep_n_frames.append(len(actions_raw[::2]))

# Verify train/val split
train_frames_total = sum(all_ep_n_frames[:cfg.n_train_episodes])
val_frames_total = sum(all_ep_n_frames[cfg.n_train_episodes:])
assert train_frames_total == train_emb.shape[0], f'Train mismatch: {train_frames_total} vs {train_emb.shape[0]}'
assert val_frames_total == val_emb.shape[0], f'Val mismatch: {val_frames_total} vs {val_emb.shape[0]}'
print(f'Train: {train_frames_total} frames in {cfg.n_train_episodes} episodes')
print(f'Val:   {val_frames_total} frames in {len(parquet_files) - cfg.n_train_episodes} episodes')

# Build episode-to-embedding-index mapping
train_offset = 0
val_offset = 0
ep_to_emb_start = {}

for ep_idx in range(len(parquet_files)):
    n_frames = all_ep_n_frames[ep_idx]
    if ep_idx < cfg.n_train_episodes:
        ep_to_emb_start[ep_idx] = ('train', train_offset)
        train_offset += n_frames
    else:
        ep_to_emb_start[ep_idx] = ('val', val_offset)
        val_offset += n_frames

def get_emb_idx(ep_idx, local_frame_idx):
    split_name, start = ep_to_emb_start[ep_idx]
    return split_name, start + local_frame_idx

print('Data loaded and verified.')


## Section 3: Predictor Classes & Utilities

In [ ]:
# Re-define Transformer predictor (same as Module A/B)
class ActionMLP(nn.Module):
    def __init__(self, action_dim=7, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(action_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, a):
        return self.net(a)


class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal_mask)
        return self.predictor(x[:, -1, :])


# Load best K=1, T=8 predictor
ckpt_path = OUTPUT_DIR / f'module_a_transformer_K1_T{cfg.best_T}.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
rollout_predictor = TransformerPredictor(
    embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
    n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
).to(device)
rollout_predictor.load_state_dict(ckpt['model_state_dict'])
rollout_predictor.eval()
print(f'Loaded Module A predictor K=1 T={cfg.best_T} (val_cos={ckpt.get("val_cos", "?")})')


In [ ]:
def cosine_loss(pred, target):
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def get_lr(step, warmup_steps, total_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

print('Utilities ready.')


## Section 4: C4 — REPA Alignment

### 4.1 Cache DINOv2 Intermediate Features

Run DINOv2 ViT-S/14 inference on all 4900 frames, extracting the CLS token at **all 12 transformer blocks**.
During REPA training, predictor layers $\{0,1,2,3\}$ are mapped to DINOv2 blocks $\{2,5,8,11\}$ (configurable via `cfg.repa_layer_map`).

This is the most computationally expensive step (~2–3h).


In [ ]:
import timm

print('Loading DINOv2 ViT-S/14 from timm...')
dino_model = timm.create_model('vit_small_patch14_reg4_dinov2.lvd142m', pretrained=True)
dino_model.eval()
dino_model = dino_model.to(device)

# Get transform for DINOv2 (224x224, ImageNet normalization)
dino_transform = timm.data.resolve_model_data_config(dino_model)
dino_transform = timm.data.create_transform(**dino_transform, is_training=False)

print(f'DINOv2 model loaded. Blocks: {len(dino_model.blocks)}')


In [ ]:
@torch.no_grad()
def extract_intermediate_features(dino_model, images_batch):
    """
    Extract post-norm CLS token at each transformer block for a batch of images.
    Returns: (B, 12, 384) tensor.
    """
    images_batch = images_batch.to(device)
    B = images_batch.shape[0]

    # Patch embedding
    x = dino_model.patch_embed(images_batch)
    x = dino_model._pos_embed(x)
    x = dino_model.norm_pre(x)

    features = torch.zeros(B, 12, cfg.embed_dim, device=device)
    for i, block in enumerate(dino_model.blocks):
        x = block(x)
        # Apply final norm to CLS so features live in the same space
        # as the predictor's output embeddings (post-norm DINOv2 CLS).
        features[:, i] = dino_model.norm(x[:, 0])

    return features.cpu()


def read_episode_frames(episode_idx, camera='agentview'):
    """Read RGB frames from the LeRobot MP4 video via PyAV. Returns (N, H, W, 3) uint8."""
    chunk = episode_idx // 1000
    ep_str = f'episode_{episode_idx:06d}'
    video_key = f'observation.images.{camera}'
    video_path = (DATA_DIR / 'robomimic-ph-lift-image' / 'videos'
                  / f'chunk-{chunk:03d}' / video_key / f'{ep_str}.mp4')
    frames = []
    container = av.open(str(video_path))
    for frame in container.decode(video=0):
        frames.append(frame.to_ndarray(format='rgb24'))
    container.close()
    return np.array(frames, dtype=np.uint8)


def cache_intermediate_features(episode_indices, output_path, ep_n_frames,
                                subsample_rate=2, batch_size=32):
    """
    Cache DINOv2 intermediate features (post-norm CLS at all 12 blocks) for every
    cached frame. Reads raw frames from MP4 videos with the same [::subsample_rate]
    subsampling used to build embeddings_{train,val}.pt, so rows line up 1:1
    with the cached CLS embeddings.

    Resumable: saves per-episode chunks so a crash mid-run doesn't restart from 0.
    """
    if output_path.exists():
        feats = torch.load(output_path, weights_only=False)
        print(f'  Loaded cached {output_path.name} ({feats.shape})')
        return feats

    part_dir = output_path.parent / f'{output_path.stem}_parts'
    part_dir.mkdir(exist_ok=True)

    all_features = []
    for ep_idx in tqdm(episode_indices, desc=f'Extracting -> {output_path.name}'):
        part_path = part_dir / f'ep_{ep_idx:04d}.pt'
        if part_path.exists():
            feats = torch.load(part_path, weights_only=False)
        else:
            frames = read_episode_frames(ep_idx)
            sub = frames[::subsample_rate]
            n_expected = ep_n_frames[ep_idx]
            sub = sub[:n_expected]
            assert len(sub) == n_expected, f'ep {ep_idx}: {len(sub)} != {n_expected} frames'
            feats_list = []
            for i in range(0, len(sub), batch_size):
                batch = [dino_transform(Image.fromarray(f)) for f in sub[i:i+batch_size]]
                batch = torch.stack(batch, dim=0)
                f = extract_intermediate_features(dino_model, batch)
                feats_list.append(f)
            feats = torch.cat(feats_list, dim=0)
            torch.save(feats, part_path)
        all_features.append(feats)

    all_features = torch.cat(all_features, dim=0)
    torch.save(all_features, output_path)
    # Clean up per-episode parts after successful merge
    for p in part_dir.glob('*.pt'):
        p.unlink()
    part_dir.rmdir()
    print(f'  Saved {output_path} ({all_features.shape})')
    return all_features


# Imports needed for video decoding (PyAV) and image conversion (PIL)
import av
from PIL import Image

# Cache DINOv2 intermediate features for train (eps 0..159) and val (eps 160..199).
# Episode ordering and [::2] subsampling match embeddings_{train,val}.pt exactly.
train_ep_idx = list(range(cfg.n_train_episodes))
val_ep_idx = list(range(cfg.n_train_episodes, len(parquet_files)))

dino_inter_train = cache_intermediate_features(
    train_ep_idx, EMBED_DIR / 'dino_intermediate_train.pt', all_ep_n_frames
)
dino_inter_val = cache_intermediate_features(
    val_ep_idx, EMBED_DIR / 'dino_intermediate_val.pt', all_ep_n_frames
)

assert dino_inter_train.shape[0] == train_emb.shape[0], 'train intermediate count mismatch'
assert dino_inter_val.shape[0] == val_emb.shape[0], 'val intermediate count mismatch'
print('DINOv2 intermediate features cached and verified.')


### 4.2 REPA Predictor Architecture

Extends `TransformerPredictor` to:
1. Return intermediate hidden states $h^{(0)}, h^{(1)}, h^{(2)}, h^{(3)}$ from each encoder layer
2. Attach projection heads $\pi_l = \mathrm{Linear}(d_\text{model}, 384)$ for each layer
3. Compute REPA loss: $\mathcal{L}_\text{REPA} = \sum_l (1 - \cos(\pi_l(h^{(l)}), f^{(l)}))$

Training loss: $\mathcal{L} = \mathcal{L}_\text{main}(\hat{z}, z^*) + \lambda \cdot \mathcal{L}_\text{REPA}$


In [ ]:
class REPAPredictor(nn.Module):
    """Transformer predictor with REPA-style intermediate alignment.
    
    Returns intermediate hidden states from each encoder layer for alignment
    with cached DINOv2 features.
    """
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2,
                 dino_layer_map=None):
        super().__init__()
        self.d_model = d_model
        self.n_layers = n_layers
        self.dino_layer_map = dino_layer_map or [2, 5, 8, 11]
        assert len(self.dino_layer_map) == n_layers
        
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        
        # REPA projection heads: one per predictor layer
        self.proj_heads = nn.ModuleList([
            nn.Linear(d_model, embed_dim) for _ in range(n_layers)
        ])
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions, return_intermediate=False):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        
        # Manually step through encoder layers to capture intermediates
        intermediates = []
        for layer in self.transformer.layers:
            x = layer(x, src_mask=causal_mask)
            intermediates.append(x[:, -1, :])  # last token at each layer
        
        pred = self.predictor(x[:, -1, :])
        
        if return_intermediate:
            return pred, intermediates
        return pred

    def compute_repa_loss(self, intermediates, dino_features):
        """
        Compute REPA alignment loss.
        intermediates: list of (B, d_model) from each predictor layer
        dino_features: (B, 12, 384) or (B, n_dino_layers, 384)
        """
        total_loss = 0.0
        for l, h in enumerate(intermediates):
            dino_l = self.dino_layer_map[l]
            if dino_l >= dino_features.shape[1]:
                continue
            f_dino = dino_features[:, dino_l, :].to(h.device).float()
            proj = self.proj_heads[l](h).float()
            total_loss += (1.0 - F.cosine_similarity(proj, f_dino, dim=-1)).mean()
        return total_loss / len(intermediates)


class REPADataset(Dataset):
    """Extended ContextTripletDataset that also returns DINOv2 intermediate features."""
    def __init__(self, embed_path, action_path, triplet_path, dino_inter_path, T=8):
        self.embeddings = torch.load(embed_path)
        self.actions = torch.load(action_path)
        self.triplets = torch.load(triplet_path)
        self.dino_inter = torch.load(dino_inter_path)
        self.T = T
        valid = self.triplets[:, 0] >= (T - 1)
        self.triplets = self.triplets[valid]
        print(f'REPA dataset: {len(self.triplets)} samples (T={T})')

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        t_idx, a_idx, tpK_idx = self.triplets[idx]
        z_seq = self.embeddings[t_idx - self.T + 1 : t_idx + 1]
        a_seq = self.actions[t_idx - self.T + 1 : t_idx + 1]
        target = self.embeddings[tpK_idx]
        dino_feat = self.dino_inter[tpK_idx]  # (12, 384) -- align to PREDICTED frame
        if z_seq.shape[0] < self.T:
            pad = self.T - z_seq.shape[0]
            z_seq = torch.cat([torch.zeros(pad, self.embeddings.shape[1]), z_seq], dim=0)
            a_seq = torch.cat([torch.zeros(pad, self.actions.shape[1]), a_seq], dim=0)
        return z_seq, a_seq, target, dino_feat

print('REPA models and dataset ready.')


In [ ]:
@torch.no_grad()
def repa_compute_val_cosine(model, loader):
    model.eval()
    cs_list = []
    for z_seq, a_seq, target, _ in loader:
        z_seq = z_seq.to(device); a_seq = a_seq.to(device)
        target = target.to(device)
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                pred = model(z_seq, a_seq, return_intermediate=False)
        else:
            pred = model(z_seq, a_seq, return_intermediate=False)
        cs = F.cosine_similarity(pred, target, dim=-1).mean().item()
        cs_list.append(cs)
    return np.mean(cs_list)


def repa_train_one_epoch(model, loader, optimizer, scaler, global_step, total_steps, lambda_val):
    model.train()
    epoch_loss, epoch_cos, epoch_repa = 0.0, 0.0, 0.0
    for z_seq, a_seq, target, dino_feat in loader:
        z_seq = z_seq.to(device); a_seq = a_seq.to(device)
        target = target.to(device); dino_feat = dino_feat.to(device)
        lr = get_lr(global_step, cfg.warmup_steps, total_steps, cfg.lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                pred, intermediates = model(z_seq, a_seq, return_intermediate=True)
                cos_loss = cosine_loss(pred, target)
                repa_loss = model.compute_repa_loss(intermediates, dino_feat)
                loss = cos_loss + lambda_val * repa_loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred, intermediates = model(z_seq, a_seq, return_intermediate=True)
            cos_loss = cosine_loss(pred, target)
            repa_loss = model.compute_repa_loss(intermediates, dino_feat)
            loss = cos_loss + lambda_val * repa_loss
            loss.backward()
            optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        epoch_cos += cos_loss.item()
        epoch_repa += repa_loss.item()
        global_step += 1
    n = len(loader)
    return epoch_loss / n, epoch_cos / n, epoch_repa / n, global_step


def repa_train_model(model, train_ldr, val_ldr, lambda_val, epochs=None, verbose=True):
    epochs = epochs or cfg.repa_epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type=='cuda'))
    total_steps = epochs * len(train_ldr)
    gs = 0
    train_losses, val_cos_list = [], []
    best_cos = -1.0
    best_state = None
    for epoch in range(1, epochs + 1):
        avg_loss, avg_cos, avg_repa, gs = repa_train_one_epoch(
            model, train_ldr, optimizer, scaler, gs, total_steps, lambda_val)
        val_cos = repa_compute_val_cosine(model, val_ldr)
        train_losses.append(avg_loss)
        val_cos_list.append(val_cos)
        if val_cos > best_cos:
            best_cos = val_cos
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch == 1 or epoch == epochs):
            print(f'E {epoch:3d}/{epochs} | loss={avg_loss:.4f} cos={1-avg_cos:.4f} repa={avg_repa:.4f} | val_cos={val_cos:.4f} | best={best_cos:.4f}')
    if best_state:
        model.load_state_dict(best_state)
    return {
        'model': model, 'val_cos': val_cos_list[-1], 'best_cos': best_cos,
        'train_losses': train_losses, 'val_cos_list': val_cos_list,
        'lambda': lambda_val
    }

print('REPA training utilities ready.')


In [ ]:
# REPA lambda sweep (warm-started from Module A predictor)
repa_results = {}

# Check if intermediate features exist
dino_train_path = EMBED_DIR / 'dino_intermediate_train.pt'
dino_val_path = EMBED_DIR / 'dino_intermediate_val.pt'

# Load Module A predictor weights for warm-start (shared by all lambda runs)
ckpt_path = OUTPUT_DIR / f'module_a_transformer_K1_T{cfg.best_T}.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'Loaded Module A predictor for warm-start (val_cos={ckpt.get("val_cos", "?")})')

if dino_train_path.exists() and dino_val_path.exists():
    print('DINOv2 intermediate features found. Starting REPA training...')
    print('=' * 60)

    for lam in cfg.repa_lambda_values:
        label = f'lambda={lam}' + (' (control, no REPA)' if lam == 0 else '')
        print(f'\nTraining predictor with {label}...')
        print('-' * 50)

        # Build loaders
        train_ds = REPADataset(
            EMBED_DIR / 'embeddings_train.pt',
            EMBED_DIR / 'actions_train.pt',
            EMBED_DIR / f'triplets_train_K1.pt',
            dino_train_path, T=cfg.best_T
        )
        val_ds = REPADataset(
            EMBED_DIR / 'embeddings_val.pt',
            EMBED_DIR / 'actions_val.pt',
            EMBED_DIR / f'triplets_val_K1.pt',
            dino_val_path, T=cfg.best_T
        )
        train_ldr = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                               num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
        val_ldr = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                             num_workers=0, pin_memory=(device.type=='cuda'))

        model = REPAPredictor(
            embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
            d_model=cfg.d_model, n_layers=cfg.n_layers_tr,
            n_heads=cfg.n_heads, dropout=cfg.dropout_p,
            dino_layer_map=cfg.repa_layer_map,
        ).to(device)

        # Warm-start from Module A predictor (proj_heads stay fresh-init)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)

        result = repa_train_model(model, train_ldr, val_ldr, lambda_val=lam,
                                  epochs=cfg.repa_epochs)
        repa_results[lam] = result

        # Save checkpoint
        ckpt_out = {
            'model_state_dict': model.state_dict(),
            'val_cos': result['val_cos'], 'best_cos': result['best_cos'],
            'lambda': lam, 'layer_map': cfg.repa_layer_map,
        }
        torch.save(ckpt_out, OUTPUT_DIR / f'module_c_repa_lambda{lam}.pt')
        print(f'  Saved outputs/module_c_repa_lambda{lam}.pt (best_cos={result["best_cos"]:.4f})')

else:
    print('DINOv2 intermediate features not found.')
    print('Run the feature caching cell above first:')
    print(f'  Expected: {dino_train_path}')
    print(f'  Expected: {dino_val_path}')
    print('Skipping REPA training for now.')


In [ ]:
if repa_results:
    fig, ax = plt.subplots(figsize=(9, 5))

    # Baseline: Module A predictor val_cos (no additional training)
    baseline_cos = ckpt.get('val_cos', 0.9518)
    ax.axhline(y=baseline_cos, color='gray', linestyle='--', linewidth=1.5,
               label=f'Module A (no addtl training): {baseline_cos:.4f}')

    colors = {0.0: '#7f8c8d', 0.1: '#e74c3c', 0.3: '#2ecc71', 0.5: '#3498db'}
    for lam in sorted(repa_results.keys()):
        r = repa_results[lam]
        lbl = f'lambda={lam}' + (' (control)' if lam == 0 else '')
        ax.plot(r['val_cos_list'], color=colors.get(lam, '#9b59b6'), linewidth=2,
                label=f'{lbl} (best={r["best_cos"]:.4f})')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Cosine Similarity')
    ax.set_title('C4: REPA Training Curves (Warm-Started from Module A)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c4_repa_training.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Summary
    print(f'\n{"lambda":<8}{"Best CosSim":<14}{"Final CosSim":<14}{"delta baseline":<14}')
    print('-' * 50)
    for lam in sorted(repa_results.keys()):
        r = repa_results[lam]
        delta = r['best_cos'] - baseline_cos
        print(f'{lam:<8}{r["best_cos"]:<14.4f}{r["val_cos"]:<14.4f}{delta:<+14.4f}')
else:
    print('No REPA results to plot (intermediate features not cached).')


## Section 5: C5 — REPA Ablation Results

Compare baseline predictor (no REPA) vs best-λ REPA predictor on three axes:
1. **Cosine similarity** on val triplets at $K \in \{1,5,10,20\}$
2. **Rollout drift**: autoregressive rollout cos_sim vs step
3. **CEM success**: flat CEM planning performance


In [ ]:
# Load baseline (Module A checkpoint) and best-REPA predictor for C5 comparison
# Always load baseline
baseline_model = TransformerPredictor(
    embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
    n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
).to(device)
baseline_model.load_state_dict(ckpt['model_state_dict'])
baseline_model.eval()
print('Loaded baseline (Module A predictor, no additional training).')

repa_model = None
control_model = None

if repa_results:
    # Load lambda=0 control (warm-started, trained without REPA loss)
    if 0 in repa_results:
        control_model = REPAPredictor(
            embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
            d_model=cfg.d_model, n_layers=cfg.n_layers_tr,
            n_heads=cfg.n_heads, dropout=cfg.dropout_p,
            dino_layer_map=cfg.repa_layer_map,
        ).to(device)
        control_ckpt = torch.load(OUTPUT_DIR / 'module_c_repa_lambda0.pt',
                                  map_location=device, weights_only=False)
        control_model.load_state_dict(control_ckpt['model_state_dict'])
        control_model.eval()
        print(f'Loaded lambda=0 control (warm-started, no REPA, best_cos={control_ckpt["best_cos"]:.4f}).')

    # Find best lambda (exclude lambda=0 control)
    repa_lams = [l for l in repa_results if l > 0]
    if repa_lams:
        best_lam = max(repa_lams, key=lambda k: repa_results[k]['best_cos'])
        print(f'Best REPA lambda: {best_lam} (cos={repa_results[best_lam]["best_cos"]:.4f})')

        repa_model = REPAPredictor(
            embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
            d_model=cfg.d_model, n_layers=cfg.n_layers_tr,
            n_heads=cfg.n_heads, dropout=cfg.dropout_p,
            dino_layer_map=cfg.repa_layer_map,
        ).to(device)
        repa_ckpt = torch.load(OUTPUT_DIR / f'module_c_repa_lambda{best_lam}.pt',
                              map_location=device, weights_only=False)
        repa_model.load_state_dict(repa_ckpt['model_state_dict'])
        repa_model.eval()
        print(f'Loaded REPA predictor (lambda={best_lam}).')
    else:
        best_lam = None
        print('No REPA (lambda>0) results available.')
else:
    best_lam = None
    print('Skipping C5: REPA results not available.')


In [ ]:
# Evaluation utility: compute val cosine similarity for a given K
@torch.no_grad()
def eval_val_cos(model, K, T=8, is_repa=False):
    val_ds = ContextTripletDataset(
        EMBED_DIR / 'embeddings_val.pt',
        EMBED_DIR / 'actions_val.pt',
        EMBED_DIR / f'triplets_val_K{K}.pt', T=T
    )
    val_ldr = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=0, pin_memory=(device.type=='cuda'))
    cs_list = []
    for z_seq, a_seq, target in val_ldr:
        z_seq = z_seq.to(device); a_seq = a_seq.to(device); target = target.to(device)
        if is_repa:
            pred = model(z_seq, a_seq, return_intermediate=False)
        else:
            pred = model(z_seq, a_seq)
        cs = F.cosine_similarity(pred, target, dim=-1).mean().item()
        cs_list.append(cs)
    return np.mean(cs_list)


# Need ContextTripletDataset for evaluation
class ContextTripletDataset(Dataset):
    def __init__(self, embed_path, action_path, triplet_path, T=1):
        self.embeddings = torch.load(embed_path)
        self.actions = torch.load(action_path)
        self.triplets = torch.load(triplet_path)
        self.T = T
        valid = self.triplets[:, 0] >= (T - 1)
        self.triplets = self.triplets[valid]
    def __len__(self): return len(self.triplets)
    def __getitem__(self, idx):
        t_idx, a_idx, tpK_idx = self.triplets[idx]
        z_seq = self.embeddings[t_idx - self.T + 1 : t_idx + 1]
        a_seq = self.actions[t_idx - self.T + 1 : t_idx + 1]
        target = self.embeddings[tpK_idx]
        if z_seq.shape[0] < self.T:
            pad = self.T - z_seq.shape[0]
            z_seq = torch.cat([torch.zeros(pad, self.embeddings.shape[1]), z_seq], dim=0)
            a_seq = torch.cat([torch.zeros(pad, self.actions.shape[1]), a_seq], dim=0)
        return z_seq, a_seq, target


if repa_model is not None:
    print('Cosine similarity comparison (baseline vs REPA)...')
    results_cos = {'K': [], 'baseline': [], 'repa': []}
    for K in cfg.K_values:
        cs_base = eval_val_cos(baseline_model, K, cfg.best_T, is_repa=False)
        cs_repa = eval_val_cos(repa_model, K, cfg.best_T, is_repa=True)
        results_cos['K'].append(K)
        results_cos['baseline'].append(cs_base)
        results_cos['repa'].append(cs_repa)
        print(f'  K={K:2d}: baseline={cs_base:.4f}  REPA={cs_repa:.4f}  Δ={cs_repa-cs_base:+.4f}')

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(cfg.K_values))
    w = 0.35
    ax.bar(x - w/2, results_cos['baseline'], w, label='Baseline (no REPA)', color='gray')
    ax.bar(x + w/2, results_cos['repa'], w, label=f'REPA (λ={best_lam})', color='steelblue')
    ax.set_xticks(x); ax.set_xticklabels([f'K={k}' for k in cfg.K_values])
    ax.set_ylabel('Val Cosine Similarity')
    ax.set_title('C5: Cosine Similarity — Baseline vs REPA')
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c5_cos_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping: REPA predictor not available.')


In [ ]:
@torch.no_grad()
def autoregressive_rollout(model, z_start, actions, K, T, max_steps=20, is_repa=False):
    """Rollout supporting both regular and REPA predictors."""
    model.eval()
    z_context = z_start.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
    a_context = actions[:1].unsqueeze(0).repeat(1, T, 1).to(device)
    pred_embs, true_embs, cos_sims = [z_start.cpu().numpy()], [z_start.cpu().numpy()], [1.0]
    current_step = 0
    for m in range(1, max_steps + 1):
        target_step = current_step + K
        if target_step >= len(actions):
            break
        if is_repa:
            pred = model(z_context.to(device), a_context.to(device), return_intermediate=False)
        else:
            pred = model(z_context.to(device), a_context.to(device))
        z_context = torch.cat([z_context[:, 1:, :], pred.unsqueeze(1)], dim=1)
        a_new = actions[target_step:target_step+1].unsqueeze(0).repeat(1, T, 1).to(device)
        a_context = a_new
        pred_embs.append(pred.squeeze(0).cpu().numpy())
        true_embs.append(val_emb[target_step].cpu().numpy())
        cs = F.cosine_similarity(pred.squeeze(0), val_emb[target_step].to(device), dim=0).item()
        cos_sims.append(cs)
        current_step = target_step
    return pred_embs, true_embs, cos_sims


# Run rollout comparison
if repa_model is not None:
    print('Rollout drift comparison...')
    start_idx = 0
    z_0 = val_emb[start_idx]
    actions_seq = val_act[start_idx:start_idx + 4 * 20 + cfg.best_T]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    for K in [1, 5]:
        _, _, cos_base = autoregressive_rollout(baseline_model, z_0, actions_seq,
                                                K=K, T=cfg.best_T, max_steps=10, is_repa=False)
        _, _, cos_repa = autoregressive_rollout(repa_model, z_0, actions_seq,
                                                K=K, T=cfg.best_T, max_steps=10, is_repa=True)
        ax.plot(cos_base, 'o-', markersize=4, linewidth=1.5,
                label=f'Baseline K={K}')
        ax.plot(cos_repa, 's--', markersize=4, linewidth=1.5,
                label=f'REPA K={K}')
    
    ax.set_xlabel('Rollout step')
    ax.set_ylabel('Cosine Similarity')
    ax.set_title('C5: Rollout Drift — Baseline vs REPA')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c5_rollout_drift.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping: REPA predictor not available.')


In [ ]:
# Flat CEM planner (adapted from Module B)
@torch.no_grad()
def flat_cem_plan(z_start, z_target, predictor, horizon, K, T,
                  n_pop=128, elite_frac=0.1, n_iters=5, noise_scale=0.5, is_repa=False):
    n_elite = max(1, int(n_pop * elite_frac))
    mu = torch.zeros(horizon, cfg.action_dim, device=device)
    sigma = torch.ones(horizon, cfg.action_dim, device=device) * noise_scale
    best_seq, best_score = None, -float('inf')
    
    for it in range(n_iters):
        eps = torch.randn(n_pop, horizon, cfg.action_dim, device=device)
        actions = mu.unsqueeze(0) + sigma.unsqueeze(0) * eps
        scores = torch.zeros(n_pop, device=device)
        
        for i in range(n_pop):
            z_ctx = z_start.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
            a_ctx = torch.zeros(1, T, cfg.action_dim, device=device)
            a_ctx[:, -1, :] = actions[i, 0]
            for step in range(horizon):
                a_cur = actions[i, step:step+1].unsqueeze(0)
                a_ctx = torch.cat([a_ctx[:, 1:, :], a_cur], dim=1)
                if is_repa:
                    pred = predictor(z_ctx, a_ctx, return_intermediate=False)
                else:
                    pred = predictor(z_ctx, a_ctx)
                z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            z_pred = z_ctx[:, -1, :].squeeze(0)
            scores[i] = F.cosine_similarity(z_pred.unsqueeze(0), z_target.unsqueeze(0), dim=-1)
        
        elite_idx = torch.topk(scores, n_elite).indices
        ea = actions[elite_idx]
        mu = ea.mean(dim=0)
        sigma = ea.std(dim=0) + 1e-6
        iter_best = scores[elite_idx[0]].item()
        if iter_best > best_score:
            best_score = iter_best; best_seq = actions[elite_idx[0]].cpu().clone()
    return best_seq, best_score


if repa_model is not None:
    print('CEM success comparison...')
    # Test on a few val episodes
    test_ep_indices = list(range(cfg.n_train_episodes, min(cfg.n_train_episodes + 10, len(parquet_files))))
    
    cem_results = {'episode': [], 'baseline_cos': [], 'repa_cos': []}
    for ep_idx in tqdm(test_ep_indices, desc='CEM eval'):
        split_name, start = ep_to_emb_start[ep_idx]
        emb = val_emb if split_name == 'val' else train_emb
        n_frames = all_ep_n_frames[ep_idx]
        if n_frames < 20:
            continue
        z_start = emb[start].to(device)
        z_target = emb[start + n_frames - 1].to(device)
        
        _, score_base = flat_cem_plan(z_start, z_target, baseline_model,
                                      horizon=20, K=1, T=cfg.best_T, is_repa=False)
        _, score_repa = flat_cem_plan(z_start, z_target, repa_model,
                                      horizon=20, K=1, T=cfg.best_T, is_repa=True)
        cem_results['episode'].append(ep_idx)
        cem_results['baseline_cos'].append(score_base)
        cem_results['repa_cos'].append(score_repa)
    
    avg_base = np.mean(cem_results['baseline_cos'])
    avg_repa = np.mean(cem_results['repa_cos'])
    print(f'\nAvg CEM goal cos: baseline={avg_base:.4f}  REPA={avg_repa:.4f}')
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(['Baseline', 'REPA'], [avg_base, avg_repa], color=['gray', 'steelblue'])
    ax.set_ylabel('Avg CEM Goal Cosine Similarity')
    ax.set_title('C5: CEM Planning Success')
    for i, v in enumerate([avg_base, avg_repa]):
        ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
    ax.set_ylim(0, 1); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'c5_cem_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping: REPA predictor not available.')


In [ ]:
print('=' * 60)
print('Module C (C4-C5) Complete')
print('=' * 60)

print(f'\nCheckpoints saved:')
for f in sorted(OUTPUT_DIR.glob('module_c_repa*.pt')):
    print(f'  {f.name}')

print(f'\nFigures saved:')
for f in sorted(OUTPUT_DIR.glob('c[4-5]_*.png')):
    print(f'  {f.name}')
